In [ ]:
# -*- coding: utf-8 -*-
from __future__ import annotations

import os
from typing import Any, Callable, Dict, Optional, Union, Iterable, Tuple

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

from pathlib import Path
from stats.masking import mask_feature
from stats.filtering import apply_filters, FilterValue

# =========================================================
# 2) 설정 (여기만 수정하면 됨)
# =========================================================
CSV_PATH = r"csv\통계용\2025-12-26_01-39_logodds_V_No_by_ASP_No_ID.csv"

# 컬럼명(파일마다 바뀌면 여기만 바꾸면 됨)
OUTCOME_COL = "outcome_level"
ASP_COL = "ASP_No"

# 사용할 상표현(ASP_No)
ASP_SET = [101, 102, 111, 112, 123, 122, 124, 113, 125, 126, 132, 131]

# outcome_level 조건
OUTCOME_MIN = 1000
OUTCOME_EXCLUDE = {2999, 1009, 4999}

# ✅ 필터는 여기 한 번만
FILTERS: Dict[str, FilterValue] = {
    ASP_COL: ASP_SET,
    OUTCOME_COL: lambda s: (s >= OUTCOME_MIN) & (~s.isin(list(OUTCOME_EXCLUDE))),

    # ---- 원하는 필터 있으면 여기 추가 ----
    # "category": ["강의", "낭독"],
    # "outcome_total": lambda s: s >= 500,
}

# feature로 쓸 컬럼
FEATURE_COL = "log_odds"   # or "z"

# 마스킹 옵션(원하면 켜기)
MASK_OPTS = dict(
    p_col="p_value",
    p_max=None,                    # 예: 0.05
    count_col="a_in_context_and_level",
    count_min=None,                # 예: 5
    abs_min=None,                  # 예: 0.2
    require_notna=None,            # 예: ["log_odds", "p_value"]
    fill_value=0.0,
)

# outcome_level(행) 단위 추가 필터: 0 아닌 feature가 너무 적으면 제거
MIN_NONZERO_FEATURES: Optional[int] = None     # 예: 3, 4

# k 여러 개를 한 번에 + k마다 여러 번 반복
K_VALUES = list(range(2, 21))   # 2~20
N_REPEATS_PER_K = 30
BASE_RANDOM_STATE = 42

# 출력
OUT_DIR = "cluster_multi_k"
os.makedirs(OUT_DIR, exist_ok=True)
OUT_BEST_BY_K = os.path.join(OUT_DIR, "best_run_by_k.csv")
OUT_LABEL_WIDE = os.path.join(OUT_DIR, "labels_wide_by_k.csv")
OUT_FEATURE_MATRIX = os.path.join(OUT_DIR, "feature_matrix_X.csv")


# =========================================================
# 3) feature matrix 만들기 (필터 1번 + 마스킹 + pivot)
# =========================================================
def build_feature_matrix(df: pd.DataFrame) -> pd.DataFrame:
    # (1) 필터는 적용
    df2 = apply_filters(df, FILTERS) #stats/filtering.py 참조

    # (2) 마스킹된 feature 만들기
    df2["_feat"] = mask_feature(df2, FEATURE_COL, **MASK_OPTS) #stats/masking.py 참조

    # (3) pivot: outcome_level × ASP_No
    if OUTCOME_COL not in df2.columns:
        raise KeyError(f"Missing OUTCOME_COL: {OUTCOME_COL}")
    if ASP_COL not in df2.columns:
        raise KeyError(f"Missing ASP_COL: {ASP_COL}")

    X = df2.pivot(index=OUTCOME_COL, columns=ASP_COL, values="_feat")

    # (4) ASP 열 고정 + 결측은 0(중립)
    X = X.reindex(columns=ASP_SET).fillna(0.0)

    # (5) (선택) 정보 적은 outcome_level 제거
    if MIN_NONZERO_FEATURES is not None:
        keep = (X != 0).sum(axis=1) >= int(MIN_NONZERO_FEATURES)
        X = X.loc[keep]

    return X


# =========================================================
# 4) k별 반복 실행 → silhouette 최대 run 선택
# =========================================================
def best_kmeans_for_k(X_scaled: np.ndarray, k: int, n_repeats: int, base_seed: int) -> Dict[str, Any]:
    best = {"silhouette": -np.inf, "labels": None, "seed": None, "inertia": None}

    for r in range(n_repeats):
        seed = base_seed + 1000 * k + r
        model = KMeans(n_clusters=k, random_state=seed, n_init=20)
        labels = model.fit_predict(X_scaled)

        try:
            score = silhouette_score(X_scaled, labels)
        except ValueError:
            continue

        if score > best["silhouette"]:
            best["silhouette"] = float(score)
            best["labels"] = labels
            best["seed"] = seed
            best["inertia"] = float(model.inertia_)

    if best["labels"] is None:
        raise RuntimeError(f"All repeats failed for k={k} (silhouette could not be computed).")

    return best


def run_multi_k(X: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X.values)

    best_rows = []
    label_wide = pd.DataFrame(index=X.index)

    for k in K_VALUES:
        if k < 2:
            continue
        if k >= X.shape[0]:   # 군집 수는 데이터 수보다 작아야 함
            continue

        best = best_kmeans_for_k(X_scaled, k, N_REPEATS_PER_K, BASE_RANDOM_STATE)
        labels = best["labels"]
        sil = best["silhouette"]

        # 비교용 wide 라벨
        label_wide[f"cluster_k{k}"] = labels

        # 상세 결과(라벨 + feature)
        out = X.copy()
        out.insert(0, "cluster", labels)
        out.insert(1, "k", k)
        out.insert(2, "best_silhouette", sil)
        out.insert(3, "best_seed", best["seed"])
        out.insert(4, "best_inertia", best["inertia"])
        out.to_csv(os.path.join(OUT_DIR, f"cluster_outcome_level_k{k}.csv"), encoding="utf-8-sig")

        # 요약(클러스터 평균 프로필)
        summary = out.groupby("cluster")[X.columns].mean()
        summary.insert(0, "size", out.groupby("cluster").size())
        summary.insert(1, "k", k)
        summary.insert(2, "best_silhouette", sil)
        summary.to_csv(os.path.join(OUT_DIR, f"cluster_summary_k{k}.csv"), encoding="utf-8-sig")

        best_rows.append({
            "k": k,
            "best_silhouette": sil,
            "best_seed": best["seed"],
            "best_inertia": best["inertia"],
            "n_outcome_levels": int(X.shape[0]),
            "n_features": int(X.shape[1]),
            "feature_col": FEATURE_COL,
        })

        print(f"[OK] k={k} best_silhouette={sil:.4f}")

    best_df = pd.DataFrame(best_rows).sort_values("k")
    return best_df, label_wide


# =========================================================
# 5) main
# =========================================================
def main():
    df = pd.read_csv(CSV_PATH)

    X = build_feature_matrix(df)
    if X.shape[0] < 2:
        raise ValueError("필터/전처리 후 outcome_level이 너무 적어. (최소 2개 필요)")

    # 참고용: feature matrix 저장
    X.to_csv(OUT_FEATURE_MATRIX, encoding="utf-8-sig")

    best_df, label_wide = run_multi_k(X)

    best_df.to_csv(OUT_BEST_BY_K, index=False, encoding="utf-8-sig")
    label_wide.to_csv(OUT_LABEL_WIDE, encoding="utf-8-sig")

    print("\n[Done]")
    print(f"- X: {OUT_FEATURE_MATRIX}")
    print(f"- best table: {OUT_BEST_BY_K}")
    print(f"- labels wide: {OUT_LABEL_WIDE}")
    print(f"- per-k outputs: {OUT_DIR}/cluster_outcome_level_k*.csv")
    print(f"- per-k summaries: {OUT_DIR}/cluster_summary_k*.csv")


if __name__ == "__main__":
    main()

[OK] k=2 best_silhouette=0.2826
[OK] k=3 best_silhouette=0.1576
[OK] k=4 best_silhouette=0.1622
[OK] k=5 best_silhouette=0.1621
[OK] k=6 best_silhouette=0.1689
[OK] k=7 best_silhouette=0.1730
[OK] k=8 best_silhouette=0.1636
[OK] k=9 best_silhouette=0.1749
[OK] k=10 best_silhouette=0.1795
[OK] k=11 best_silhouette=0.1866
[OK] k=12 best_silhouette=0.1854
[OK] k=13 best_silhouette=0.1869
[OK] k=14 best_silhouette=0.1823
[OK] k=15 best_silhouette=0.1833
[OK] k=16 best_silhouette=0.1857
[OK] k=17 best_silhouette=0.1879
[OK] k=18 best_silhouette=0.1888
[OK] k=19 best_silhouette=0.1891
[OK] k=20 best_silhouette=0.1910

[Done]
- X: cluster_multi_k\feature_matrix_X.csv
- best table: cluster_multi_k\best_run_by_k.csv
- labels wide: cluster_multi_k\labels_wide_by_k.csv
- per-k outputs: cluster_multi_k/cluster_outcome_level_k*.csv
- per-k summaries: cluster_multi_k/cluster_summary_k*.csv
